# Ancient Kannada (Halegannada) to Modern Kannada (Hosagannada) Translation Model
This notebook trains a Neural Machine Translation (Seq2Seq / mT5) model to translate ancient Kannada epigraphical sentences into modern contemporary Kannada.

In [ ]:
# Step 1: Install dependencies on Colab with GPU
!pip install -q torch torchvision transformers datasets accelerate sentencepiece sacrebleu

In [ ]:
# Step 2: Upload dataset
# Upload the 'halegannada_parallel_dataset.json' generated from your project's data folder
from google.colab import files
import os
if not os.path.exists('halegannada_parallel_dataset.json'):
    print('Upload your halegannada_parallel_dataset.json file:')
    uploaded = files.upload()

In [ ]:
# Step 3: Train Seq2Seq mT5-small on GPU
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

dataset = load_dataset('json', data_files='halegannada_parallel_dataset.json')['train']
dataset = dataset.train_test_split(test_size=0.1, seed=42)

model_name = 'google/mt5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_fn(batch):
    inputs = ['translate Halegannada to Hosagannada: ' + text for text in batch['source_ancient']]
    targets = batch['target_modern']
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding='max_length')
    labels = tokenizer(targets, max_length=128, truncation=True, padding='max_length')
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = dataset['train'].map(preprocess_fn, batched=True)
tokenized_eval = dataset['test'].map(preprocess_fn, batched=True)

args = Seq2SeqTrainingArguments(
    output_dir='./halegannada_model',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    fp16=torch.cuda.is_available(),
    logging_steps=20
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

trainer.train()
model.save_pretrained('./best_translator')
tokenizer.save_pretrained('./best_translator')
print('Model training complete and saved to ./best_translator')

In [ ]:
# Step 4: Test Translation on Ancient Inscription
test_sentence = 'ಶ್ರೀಮತ್ ಕದಂಬಪನ್ ಕಾಗುಸ್ಥಭಟ್ಟೋರನ್ ಆಳೆ'
input_text = 'translate Halegannada to Hosagannada: ' + test_sentence
inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_length=128)
translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Ancient Inscription:', test_sentence)
print('Modern Kannada:     ', translated)